# Cost Analysis for Model Optimization

In this notebook, we'll analyze the cost implications of the various model optimization techniques we've explored. We'll calculate the ROI and payback period for each technique.

## 1. Import Dependencies

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display, HTML

## 2. Load Metrics from Previous Notebooks

In [ ]:
# Load baseline metrics
try:
    with open('baseline-metrics.json', 'r') as f:
        baseline_metrics = json.load(f)
    print(f"Loaded baseline metrics for {len(baseline_metrics)} models")
except FileNotFoundError:
    print("baseline-metrics.json not found. Using empty dictionary.")
    baseline_metrics = {}

# Load quantized metrics
try:
    with open('quantized-metrics.json', 'r') as f:
        quantized_metrics = json.load(f)
    print(f"Loaded quantized metrics for {len(quantized_metrics)} models")
except FileNotFoundError:
    print("quantized-metrics.json not found. Using empty dictionary.")
    quantized_metrics = {}

# Load pruned metrics
try:
    with open('pruned-metrics.json', 'r') as f:
        pruned_metrics = json.load(f)
    print(f"Loaded pruned metrics for {len(pruned_metrics)} models")
except FileNotFoundError:
    print("pruned-metrics.json not found. Using empty dictionary.")
    pruned_metrics = {}

# Load distilled metrics
try:
    with open('distilled-metrics.json', 'r') as f:
        distilled_metrics = json.load(f)
    print(f"Loaded distilled metrics for {len(distilled_metrics)} models")
except FileNotFoundError:
    print("distilled-metrics.json not found. Using empty dictionary.")
    distilled_metrics = {}

## 3. Define Cost Parameters

We'll define the cost parameters for our analysis. These are approximate values and can be adjusted based on your specific use case.

In [ ]:
# Define cost parameters
instance_costs = {
    "ml.g4dn.xlarge": 0.736,  # $ per hour
    "ml.g4dn.2xlarge": 0.94,  # $ per hour
    "ml.g5.xlarge": 1.006,    # $ per hour
    "ml.g5.2xlarge": 1.515    # $ per hour
}

# Storage cost per GB per month
storage_cost_per_gb_month = 0.023  # $ per GB per month for S3 Standard

# Implementation costs (time spent on optimization)
implementation_costs = {
    "quantization": 50,    # $ (approximately 1 hour of engineer time + compute costs)
    "pruning": 100,        # $ (approximately 2 hours of engineer time + compute costs)
    "distillation": 500    # $ (approximately 8 hours of engineer time + compute costs)
}

# Default instance type for inference
default_instance_type = "ml.g4dn.xlarge"

# Display cost parameters
print("Instance costs ($ per hour):")
for instance, cost in instance_costs.items():
    print(f"  {instance}: ${cost:.3f}")

print(f"\nStorage cost: ${storage_cost_per_gb_month:.3f} per GB per month")

print("\nImplementation costs:")
for technique, cost in implementation_costs.items():
    print(f"  {technique}: ${cost:.2f}")

## 4. Define Cost Estimation Functions

In [ ]:
def estimate_monthly_cost(model_metrics, requests_per_month=1000000):
    """Estimate monthly cost for a model based on its metrics."""
    # Calculate inference time in hours
    inference_time_hours = (model_metrics["inference_time"] * requests_per_month) / (1000 * 60 * 60)
    
    # Calculate compute cost
    instance_cost = instance_costs.get(default_instance_type, 0.736)  # Default to g4dn.xlarge if not found
    compute_cost = inference_time_hours * instance_cost
    
    # Calculate storage cost
    storage_cost = (model_metrics["model_size"] / 1024) * storage_cost_per_gb_month
    
    # Total cost
    total_cost = compute_cost + storage_cost
    
    return {
        "compute_cost": compute_cost,
        "storage_cost": storage_cost,
        "total_cost": total_cost
    }

def calculate_roi(baseline_cost, optimized_cost, implementation_cost, months=12):
    """Calculate ROI for an optimization technique."""
    monthly_savings = baseline_cost - optimized_cost
    total_savings = monthly_savings * months
    roi = (total_savings - implementation_cost) / implementation_cost if implementation_cost > 0 else float('inf')
    payback_period = implementation_cost / monthly_savings if monthly_savings > 0 else float('inf')
    
    return {
        "monthly_savings": monthly_savings,
        "total_savings": total_savings,
        "roi": roi,
        "payback_period": payback_period
    }

## 5. Calculate Monthly Costs for Different Request Volumes

In [ ]:
# Define request volumes to analyze
request_volumes = [10000, 100000, 1000000, 10000000, 100000000]  # Requests per month

# Calculate monthly costs for each model and request volume
monthly_costs = {}

for model_key in baseline_metrics.keys():
    monthly_costs[model_key] = {}
    
    for requests in request_volumes:
        monthly_costs[model_key][requests] = {
            "baseline": estimate_monthly_cost(baseline_metrics[model_key], requests)["total_cost"]
        }
        
        if model_key in quantized_metrics:
            monthly_costs[model_key][requests]["quantized"] = estimate_monthly_cost(quantized_metrics[model_key], requests)["total_cost"]
        
        if model_key in pruned_metrics:
            monthly_costs[model_key][requests]["pruned"] = estimate_monthly_cost(pruned_metrics[model_key], requests)["total_cost"]
        
        if model_key in distilled_metrics:
            # For distilled models, we need to use the student metrics
            student_metrics = distilled_metrics[model_key]["student"]
            monthly_costs[model_key][requests]["distilled"] = estimate_monthly_cost(student_metrics, requests)["total_cost"]

# Create a DataFrame for easier visualization
cost_data = []

for model_key in monthly_costs.keys():
    model_name = baseline_metrics[model_key]["model_name"]
    
    for requests in request_volumes:
        for technique, cost in monthly_costs[model_key][requests].items():
            cost_data.append({
                "Model": model_name,
                "Requests per Month": requests,
                "Technique": technique,
                "Monthly Cost ($)": cost
            })

cost_df = pd.DataFrame(cost_data)

# Display the DataFrame
cost_df.pivot_table(
    index=["Model", "Technique"],
    columns="Requests per Month",
    values="Monthly Cost ($)"
).round(2)

## 6. Visualize Cost Comparison

In [ ]:
# Set up the visualization style
sns.set(style="whitegrid")
plt.figure(figsize=(12, 6))

# Choose a specific request volume for visualization
request_volume = 1000000  # 1 million requests per month

# Filter data for the chosen request volume
filtered_data = cost_df[cost_df["Requests per Month"] == request_volume]

# Create the bar chart
ax = sns.barplot(x="Model", y="Monthly Cost ($)", hue="Technique", data=filtered_data)

# Customize the chart
plt.title(f"Monthly Cost Comparison for {request_volume:,} Requests per Month")
plt.xlabel("Model")
plt.ylabel("Monthly Cost ($)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Technique")
plt.tight_layout()

# Show the chart
plt.show()

## 7. Calculate ROI and Payback Period

In [ ]:
# Calculate ROI for each optimization technique
roi_data = []

# Choose a specific request volume for ROI calculation
request_volume = 1000000  # 1 million requests per month

def calculate_break_even_requests(baseline_metrics, optimized_metrics, implementation_cost):
    """Calculate the number of requests needed to break even on the implementation cost."""
    # Calculate cost per request for each model
    baseline_inference_time = baseline_metrics["inference_time"] / 1000  # Convert to seconds
    optimized_inference_time = optimized_metrics["inference_time"] / 1000  # Convert to seconds
    
    # Calculate cost per request
    instance_cost = instance_costs.get(default_instance_type, 0.736) / 3600  # Convert to cost per second
    baseline_cost_per_request = baseline_inference_time * instance_cost
    optimized_cost_per_request = optimized_inference_time * instance_cost
    
    # Calculate savings per request
    savings_per_request = baseline_cost_per_request - optimized_cost_per_request
    
    # Calculate break-even requests
    if savings_per_request > 0:
        break_even_requests = implementation_cost / savings_per_request
        return break_even_requests
    else:
        return float('inf')  # No break-even point

# Calculate ROI for each model and technique
for model_key in baseline_metrics.keys():
    model_name = baseline_metrics[model_key]["model_name"]
    
    # Calculate baseline cost
    baseline_cost = monthly_costs[model_key][request_volume]["baseline"]
    
    # Calculate ROI for quantization
    if model_key in quantized_metrics:
        quantized_cost = monthly_costs[model_key][request_volume]["quantized"]
        roi_result = calculate_roi(
            baseline_cost,
            quantized_cost,
            implementation_costs["quantization"]
        )
        
        # Calculate break-even requests
        break_even_requests = calculate_break_even_requests(
            baseline_metrics[model_key],
            quantized_metrics[model_key],
            implementation_costs["quantization"]
        )
        
        roi_data.append({
            "Model": model_name,
            "Technique": "quantization",
            "Implementation Cost ($)": implementation_costs["quantization"],
            "Monthly Savings ($)": roi_result["monthly_savings"],
            "Annual Savings ($)": roi_result["total_savings"],
            "ROI (1 year)": roi_result["roi"],
            "Payback Period (months)": roi_result["payback_period"],
            "Break-even Requests": break_even_requests
        })
    
    # Calculate ROI for pruning
    if model_key in pruned_metrics:
        pruned_cost = monthly_costs[model_key][request_volume]["pruned"]
        roi_result = calculate_roi(
            baseline_cost,
            pruned_cost,
            implementation_costs["pruning"]
        )
        
        # Calculate break-even requests
        break_even_requests = calculate_break_even_requests(
            baseline_metrics[model_key],
            pruned_metrics[model_key],
            implementation_costs["pruning"]
        )
        
        roi_data.append({
            "Model": model_name,
            "Technique": "pruning",
            "Implementation Cost ($)": implementation_costs["pruning"],
            "Monthly Savings ($)": roi_result["monthly_savings"],
            "Annual Savings ($)": roi_result["total_savings"],
            "ROI (1 year)": roi_result["roi"],
            "Payback Period (months)": roi_result["payback_period"],
            "Break-even Requests": break_even_requests
        })
    
    # Calculate ROI for distillation
    if model_key in distilled_metrics:
        distilled_cost = monthly_costs[model_key][request_volume]["distilled"]
        roi_result = calculate_roi(
            baseline_cost,
            distilled_cost,
            implementation_costs["distillation"]
        )
        
        # Calculate break-even requests
        break_even_requests = calculate_break_even_requests(
            baseline_metrics[model_key],
            distilled_metrics[model_key]["student"],
            implementation_costs["distillation"]
        )
        
        roi_data.append({
            "Model": model_name,
            "Technique": "distillation",
            "Implementation Cost ($)": implementation_costs["distillation"],
            "Monthly Savings ($)": roi_result["monthly_savings"],
            "Annual Savings ($)": roi_result["total_savings"],
            "ROI (1 year)": roi_result["roi"],
            "Payback Period (months)": roi_result["payback_period"],
            "Break-even Requests": break_even_requests
        })

# Create a DataFrame for ROI data
roi_df = pd.DataFrame(roi_data)

# Format the DataFrame for display
display_df = roi_df.copy()
display_df["ROI (1 year)"] = display_df["ROI (1 year)"].apply(lambda x: f"{x:.2f}x" if x != float('inf') else "∞")
display_df["Payback Period (months)"] = display_df["Payback Period (months)"].apply(lambda x: f"{x:.2f}" if x != float('inf') else "∞")
display_df["Break-even Requests"] = display_df["Break-even Requests"].apply(lambda x: f"{x:,.0f}" if x != float('inf') else "∞")
display_df["Monthly Savings ($)"] = display_df["Monthly Savings ($)"].apply(lambda x: f"${x:.2f}")
display_df["Annual Savings ($)"] = display_df["Annual Savings ($)"].apply(lambda x: f"${x:.2f}")
display_df["Implementation Cost ($)"] = display_df["Implementation Cost ($)"].apply(lambda x: f"${x:.2f}")

# Display the DataFrame
display_df

## 8. Generate Recommendations

Based on the cost analysis, we'll generate recommendations for each model.

In [ ]:
def generate_recommendations(model_key, baseline_metrics, quantized_metrics, pruned_metrics, roi_df):
    """Generate recommendations for a specific model based on cost analysis."""
    model_name = baseline_metrics[model_key]["model_name"]
    task = baseline_metrics[model_key]["task"]
    
    # Filter ROI data for this model
    model_roi = roi_df[roi_df["Model"] == model_name]
    
    # Determine the best technique based on ROI
    if not model_roi.empty:
        # Convert ROI to numeric for comparison
        model_roi_numeric = model_roi.copy()
        model_roi_numeric["ROI (1 year)"] = pd.to_numeric(model_roi_numeric["ROI (1 year)"].str.replace("x", "").replace("∞", "9999"))
        
        # Get the technique with the highest ROI
        best_technique = model_roi_numeric.loc[model_roi_numeric["ROI (1 year)"].idxmax()]["Technique"]
        best_roi = model_roi_numeric.loc[model_roi_numeric["ROI (1 year)"].idxmax()]["ROI (1 year)"]
        
        # Generate recommendations
        recommendations = []
        
        # General recommendation based on best technique
        if best_roi > 0:
            recommendations.append(f"Based on ROI analysis, {best_technique} is the recommended optimization technique for this model.")
        else:
            recommendations.append("Based on ROI analysis, none of the optimization techniques provide a positive ROI for this model.")
        
        # Specific recommendations based on model characteristics and task
        if task in ["text-classification", "sequence-classification"]:
            recommendations.append("For classification tasks, quantization often provides the best balance of performance and implementation effort.")
        
        if task in ["token-classification", "question-answering"]:
            recommendations.append("For token-level tasks, pruning may preserve accuracy better than other techniques.")
        
        # Recommendations based on model size
        model_size_mb = baseline_metrics[model_key]["model_size"]
        if model_size_mb > 500:  # Large model
            recommendations.append("For large models (>500MB), consider combining techniques: quantization followed by pruning.")
        elif model_size_mb < 100:  # Small model
            recommendations.append("For small models (<100MB), the benefits of optimization may be limited unless you have very high request volumes.")
        
        # Recommendations based on request volume
        recommendations.append("If your request volume is low (<10,000/month), focus on storage optimization rather than inference speed.")
        recommendations.append("If your request volume is high (>1M/month), prioritize techniques that improve inference speed.")
        
        return recommendations
    else:
        return ["Insufficient data to generate recommendations for this model."]

# Generate recommendations for each model
for model_key in baseline_metrics.keys():
    print(f"\n=== Recommendations for {model_key} ===\n")
    recommendations = generate_recommendations(
        model_key,
        baseline_metrics,
        quantized_metrics,
        pruned_metrics,
        roi_df
    )
    
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec}")

## 9. Conclusion

In this notebook, we've analyzed the cost implications of various model optimization techniques. We've calculated the ROI and payback period for each technique, and generated recommendations based on the analysis.

### Key Takeaways:

1. **Quantization** offers the best ROI for most models due to its low implementation cost and significant performance improvements.

2. **Pruning** can provide additional benefits, especially for larger models, but requires more implementation effort.

3. **Distillation** has the highest implementation cost but can provide the most significant performance improvements for high-volume inference scenarios.

4. The optimal technique depends on your specific use case, including:
   - Model size and architecture
   - Task type (classification, token-level tasks, etc.)
   - Request volume
   - Available implementation resources

5. For production deployments, consider combining techniques (e.g., quantization followed by pruning) for maximum benefit.